# ESCI Product Relevance Classification

1. **Eksperimen klasifikasi relevansi query–produk empat kelas:**

- E = Exact
- S = Substitute
- C = Complement
- I = Irrelevant

Metodologi:

- Hanya `product_locale == "us"` dan `large_version == 1`.
- Official TRAIN/TEST dipisahkan memakai kolom asli `split`.
- TRAIN representative: 100.000 baris stratified.
- TEST/final evaluation representative: 30.551 baris stratified.
- TRAIN 100K dibagi tetap menjadi 90K training dan 10K validation.
- Random state/seed: 42.
- Test tidak digunakan untuk development, model selection, tuning, atau early stopping.

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

# ============================================================
# 1. MOUNT GOOGLE DRIVE
# ============================================================
drive.mount("/content/drive")


# ============================================================
# 2. REPOSITORY CONFIGURATION
# ============================================================
REPOSITORY_URL = "https://github.com/niciiu/esci-product-relevance.git"
BRANCH = "main"

# Repository lokal di Colab
COLAB_ROOT = Path("/content/esci-product-relevance")


# ============================================================
# 3. GOOGLE DRIVE PROJECT FOLDER
# ============================================================
# IMPORTANT:
# "esci-product-relevance (1)" di My Drive adalah SHORTCUT.
# Karena itu kita menggunakan TARGET folder shortcut secara langsung.

DRIVE_FOLDER = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/"
    "esci-product-relevance (1)"
)


# ============================================================
# 4. DATA & OUTPUT PATH
# ============================================================
RAW_DATA_DIR = DRIVE_FOLDER / "data" / "raw"

# Jangan tambahkan /semantic_bert di sini.
# run_main_experiment.py akan menambah nama experiment sendiri.
OUTPUT_DIR = DRIVE_FOLDER / "experiment_outputs"


# ============================================================
# 5. VALIDATE DRIVE PROJECT
# ============================================================
if not DRIVE_FOLDER.exists():
    raise FileNotFoundError(
        f"Folder project tidak ditemukan:\n{DRIVE_FOLDER}"
    )

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder raw dataset tidak ditemukan:\n{RAW_DATA_DIR}"
    )


# ============================================================
# 6. CLONE / UPDATE GITHUB REPOSITORY
# ============================================================
if (COLAB_ROOT / ".git").exists():

    print("Repository sudah ada. Updating repository...")

    subprocess.run(
        ["git", "-C", str(COLAB_ROOT), "fetch", "origin"],
        check=True,
    )

    subprocess.run(
        ["git", "-C", str(COLAB_ROOT), "checkout", BRANCH],
        check=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(COLAB_ROOT),
            "pull",
            "--ff-only",
            "origin",
            BRANCH,
        ],
        check=True,
    )

else:

    print("Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            BRANCH,
            REPOSITORY_URL,
            str(COLAB_ROOT),
        ],
        check=True,
    )


# ============================================================
# 7. SET WORKING DIRECTORY
# ============================================================
os.chdir(COLAB_ROOT)

if str(COLAB_ROOT) not in sys.path:
    sys.path.insert(0, str(COLAB_ROOT))


# ============================================================
# 8. INSTALL PYTHON DEPENDENCIES
# ============================================================
print("Installing requirements...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "-r",
        "requirements.txt",
    ],
    check=True,
)


# ============================================================
# 9. CREATE PERSISTENT OUTPUT DIRECTORY
# ============================================================
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 10. PRINT CONFIGURATION
# ============================================================
print("\n" + "=" * 60)
print("CONFIGURATION")
print("=" * 60)

print("Repository       :", COLAB_ROOT)
print("Drive project    :", DRIVE_FOLDER)
print("Raw data         :", RAW_DATA_DIR)
print("Persistent output:", OUTPUT_DIR)

print("=" * 60)


# ============================================================
# 11. VERIFY IMPORTANT PATHS
# ============================================================
print("\nPATH CHECK")
print("-" * 60)

print("Project exists    :", DRIVE_FOLDER.exists())
print("Raw data exists   :", RAW_DATA_DIR.exists())
print("Output exists     :", OUTPUT_DIR.exists())

print("\nRaw data files:")
for file in sorted(RAW_DATA_DIR.iterdir()):
    print("  -", file.name)

print("\nOutput folders:")
for folder in sorted(OUTPUT_DIR.iterdir()):
    if folder.is_dir():
        print("  -", folder.name)

print("\nSetup selesai.")

Mounted at /content/drive
Cloning repository...
Installing requirements...

CONFIGURATION
Repository       : /content/esci-product-relevance
Drive project    : /content/drive/.shortcut-targets-by-id/1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/esci-product-relevance (1)
Raw data         : /content/drive/.shortcut-targets-by-id/1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/esci-product-relevance (1)/data/raw
Persistent output: /content/drive/.shortcut-targets-by-id/1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/esci-product-relevance (1)/experiment_outputs

PATH CHECK
------------------------------------------------------------
Project exists    : True
Raw data exists   : True
Output exists     : True

Raw data files:
  - shopping_queries_dataset_examples.parquet
  - shopping_queries_dataset_products.parquet

Output folders:
  - lexical_concatenated
  - lexical_separate
  - semantic_bert
  - semantic_bert (1)
  - semantic_bert (10)
  - semantic_bert (11)
  - semantic_bert (12)
  - semantic_bert (13)
  - semantic_bert (14)

In [ ]:
!git -C /content/esci-product-relevance pull origin main
!git -C /content/esci-product-relevance log -1 --oneline

From https://github.com/niciiu/esci-product-relevance
 * branch            main       -> FETCH_HEAD
Already up to date.
c1e522e (HEAD -> main, origin/main, origin/HEAD) Refine semantic checkpoint handling


**2. GPU check**

Gunakan GPU untuk RoBERTa-base dan DeBERTa-v3-base.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/"
    "esci-product-relevance (1)/"
    "experiment_outputs"
)

for folder_name in [
    "semantic_bert",
    "semantic_bert (1)",
    "semantic_bert (7)",
    "semantic_bert (14)",
    "semantic_bert (15)",
    "semantic_bert (22)",
]:
    folder = OUTPUT_DIR / folder_name / "checkpoint-500"

    print("\n" + "=" * 60)
    print(folder_name)
    print("=" * 60)

    if folder.exists():
        for item in sorted(folder.iterdir()):
            if item.is_file():
                print(f"FILE : {item.name} | {item.stat().st_size / 1024 / 1024:.2f} MB")
            else:
                print(f"DIR  : {item.name}")
    else:
        print("checkpoint-500 tidak ditemukan")


semantic_bert
FILE : config.json | 0.00 MB
FILE : model.safetensors | 417.68 MB
FILE : optimizer.pt | 835.47 MB
FILE : tokenizer.json | 0.68 MB
FILE : tokenizer_config.json | 0.00 MB
FILE : training_args.bin | 0.01 MB

semantic_bert (1)
FILE : .tmpbm6NDn | 417.68 MB
FILE : config.json | 0.00 MB

semantic_bert (7)
FILE : .tmpqjUfwY | 417.68 MB
FILE : config.json | 0.00 MB

semantic_bert (14)
FILE : config.json | 0.00 MB
FILE : model.safetensors | 417.68 MB
FILE : optimizer.pt | 835.47 MB
FILE : tokenizer.json | 0.68 MB
FILE : tokenizer_config.json | 0.00 MB
FILE : training_args.bin | 0.01 MB

semantic_bert (15)
FILE : .tmplrZzW7 | 417.68 MB
FILE : config.json | 0.00 MB

semantic_bert (22)
FILE : config.json | 0.00 MB
FILE : model.safetensors | 417.68 MB
FILE : optimizer.pt | 835.47 MB
FILE : tokenizer.json | 0.68 MB
FILE : tokenizer_config.json | 0.00 MB
FILE : training_args.bin | 0.01 MB


**3. Build fixed representative subsets**

Tahap ini:

1. Membaca official ESCI US dengan `large_version == 1`.
2. Memisahkan official TRAIN dan TEST dari kolom `split`.
3. Mengambil 100.000 sampel stratified dari official TRAIN.
4. Mengambil 30.551 sampel stratified dari official TEST.
5. Membagi 100.000 representative TRAIN menjadi 90.000 training dan 10.000 validation.

Seluruh output disimpan di Google Drive.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.build_us_representative_subset",
        "--raw-data-dir",
        str(RAW_DATA_DIR),
        "--output-dir",
        str(DRIVE_FOLDER),
        "--seed",
        "42",
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'scripts.build_us_representative_subset', '--raw-data-dir', '/content/drive/MyDrive/esci-product-relevance (1)/data/raw', '--output-dir', '/content/drive/MyDrive/esci-product-relevance (1)', '--seed', '42'], returncode=0)

**4. Audit data sebelum dan setelah sampling**

Tabel berikut menunjukkan:

- total official US TRAIN, TEST, dan gabungan;
- representative TRAIN/TEST;
- training 90K, validation 10K, final test 30.551;
- persentase terhadap official population;
- persentase terhadap representative subset;
- count dan persentase label E/S/C/I.

In [ ]:
import json
import pandas as pd

REPORT_PATH = DRIVE_FOLDER / "esci_us_sampling_report.json"

with REPORT_PATH.open(encoding="utf-8") as file:
    report = json.load(file)

print("Dataset metadata")
display(pd.DataFrame([report["dataset"]]))

print("Methodology metadata")
display(pd.DataFrame([report["methodology"]]))

print("Ukuran dataset dan persentase")
size_rows = [
    {"split": split_name, **values}
    for split_name, values in report["size_audit"].items()
]
display(pd.DataFrame(size_rows).fillna(""))

print("Distribusi label E/S/C/I dalam persen")
distribution_rows = []

for split_name, values in report["split_summaries"].items():
    for label, label_values in values["label_distribution"].items():
        distribution_rows.append(
            {
                "split": split_name,
                "label": label,
                **label_values,
            }
        )

distribution_table = pd.DataFrame(distribution_rows)

display(
    distribution_table.pivot(
        index="label",
        columns="split",
        values="percentage",
    ).round(4)
)

print("Distribusi label E/S/C/I dalam jumlah baris")
display(
    distribution_table.pivot(
        index="label",
        columns="split",
        values="count",
    )
)

Dataset metadata


,product_locale,large_version,official_train_rows,official_test_rows,official_us_total_rows
0,us,1,1393063,425762,1818825


Methodology metadata


,task,labels,random_state,official_split_preserved,test_used_only_for_final_evaluation
0,four_class_query_product_relevance_classification,"{'E': 'Exact', 'S': 'Substitute', 'C': 'Comple...",42,True,True


Ukuran dataset dan persentase


,split,rows,percentage_of_official_train,percentage_of_official_test,percentage_of_official_us_total,percentage_of_representative_train,percentage_of_representative_test
0,official_us_train_before_sampling,1393063,100.0,,,,
1,official_us_test_before_sampling,425762,,100.0,,,
2,official_us_total_before_sampling,1818825,,,100.0,,
3,representative_train_after_sampling,100000,7.178426,,,,
4,representative_test_after_sampling,30551,,7.175605,,,
5,representative_combined_after_sampling,130551,,,7.177766,,
6,training_after_train_validation_split,90000,6.460584,,,90.0,
7,validation_after_train_validation_split,10000,0.717843,,,10.0,
8,final_evaluation_test,30551,,7.175605,,,100.0


Distribusi label E/S/C/I dalam persen


split,evaluation_test_after_sampling,official_test_before_sampling,official_train_before_sampling,train_subset_after_sampling,training_after_train_validation_split,validation_after_train_validation_split
label,,,,,,
C,2.5662,2.5653,2.0909,2.091,2.0911,2.09
E,65.1403,65.1416,69.6458,69.646,69.6456,69.65
I,9.9669,9.9675,8.5759,8.576,8.5767,8.57
S,22.3266,22.3256,19.6875,19.687,19.6867,19.69


Distribusi label E/S/C/I dalam jumlah baris


split,evaluation_test_after_sampling,official_test_before_sampling,official_train_before_sampling,train_subset_after_sampling,training_after_train_validation_split,validation_after_train_validation_split
label,,,,,,
C,784,10922,29127,2091,1882,209
E,19901,277348,970210,69646,62681,6965
I,3045,42438,119467,8576,7719,857
S,6821,95054,274259,19687,17718,1969


## Preprocessing

Preprocessing dilakukan otomatis oleh pipeline saat setiap eksperimen berjalan.

### Lexical: TF-IDF + Linear SVM

- Missing product text diisi string kosong.
- HTML, entity, control characters, dan whitespace berlebih dibersihkan.
- `product_title`, `product_description`, dan `product_bullet_point` digabung menjadi product text.
- Text lexical dinormalisasi: lowercase, punctuation removal, dan whitespace normalization.
- TF-IDF hanya di-fit pada training 90K.

### Semantic: RoBERTa-base dan DeBERTa-v3-base

- Missing product text, HTML, dan structural noise dibersihkan.
- Product fields digabung menjadi product text.
- Hanya whitespace normalization.
- Capitalization, punctuation, dan natural text dipertahankan untuk tokenizer transformer.
- Fine-tuning memakai training 90K saja; validation 10K dipakai untuk `eval_loss`, checkpointing, dan early stopping.



tetapi preprocessing dijalankan otomatis di dalam pipeline saat eksperimen—bukan sebagai cell Colab terpisah. Ini lebih aman agar preprocessing konsisten dan tidak ada leakage.

Di scripts.run_main_experiment.py:
training = clean_dataset(training, arm="lexical")
validation = clean_dataset(validation, arm="lexical")
test = clean_dataset(test, arm="lexical") if test is not None else None
Untuk semantic:
training = clean_dataset(training, arm="semantic")
validation = clean_dataset(validation, arm="semantic")

test = clean_dataset(test, arm="semantic") if test is not None else None
Preprocessing-nya:
Model	Preprocessing
TF-IDF + Linear SVM	Missing text menjadi string kosong, HTML/noise dibersihkan, product title + description + bullet point digabung, lalu lowercase, punctuation removal, dan normalisasi whitespace.
RoBERTa / DeBERTa	Missing text dan HTML/noise dibersihkan, product text digabung, whitespace dinormalisasi saja. Huruf besar, punctuation, dan natural language tetap dipertahankan.


Input yang digunakan hanya:
query
product_title
product_description
product_bullet_point
Untuk lexical, vectorizer baru di-fit menggunakan training 90K:
X_train = vectorizer.fit_transform(train_text)
X_validation = vectorizer.transform(validation_text)
X_test = vectorizer.transform(test_text)
Jadi validation dan test tidak ikut membentuk vocabulary/IDF TF-IDF.

**5. Development — Lexical models**

Menjalankan dua model lexical:

1. TF-IDF Separate + Linear SVM
2. TF-IDF Concatenated + Linear SVM

Pada tahap ini, script hanya membaca training 90K dan validation 10K. Test 30.551 tidak dibaca.

In [ ]:
LEXICAL_EXPERIMENTS = [
    "lexical_separate",
    "lexical_concatenated",
]

for experiment in LEXICAL_EXPERIMENTS:
    print(f"\n{'=' * 70}")
    print(f"Development: {experiment}")
    print(f"{'=' * 70}")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "scripts.run_main_experiment",
            "--config",
            "configs/main_experiment.yaml",
            "--data-dir",
            str(DRIVE_FOLDER),
            "--output-dir",
            str(OUTPUT_DIR),
            "--experiment",
            experiment,
        ],
        check=True,
    )


Development: lexical_separate

Development: lexical_concatenated


## Semantic Model: BERT-base

Eksperimen semantic menggunakan model pre-trained **BERT-base uncased**
(`bert-base-uncased`) untuk melakukan klasifikasi relevansi query–produk
ke dalam empat kelas ESCI: Exact, Substitute, Complement, dan Irrelevant.

Input model berupa pasangan teks:

- Query pengguna
- Informasi produk: product title, product description, dan bullet point

Konfigurasi utama:

- Maksimum epoch: **10**
- Early stopping: training berhenti jika validation loss tidak membaik
  selama **2 epoch berturut-turut**
- Checkpoint disimpan setiap **500 step**
- Progress training ditampilkan per step dan dalam persentase epoch
- Checkpoint disimpan secara persisten di Google Drive

Jika runtime Colab berhenti, jalankan kembali sel training BERT dengan
`--experiment semantic_bert`. Sistem akan otomatis melanjutkan dari
checkpoint lengkap terakhir di folder:

`experiment_outputs/semantic_bert/`

Data test tidak dibaca pada tahap ini. Evaluasi test hanya dilakukan
setelah seluruh keputusan eksperimen ditetapkan.

In [ ]:
from pathlib import Path

BERT_OUTPUT_DIR = OUTPUT_DIR / "semantic_bert"
BERT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Folder BERT:", BERT_OUTPUT_DIR)

Folder BERT: /content/drive/.shortcut-targets-by-id/1SIJTwO7lCQASwhVK7vL3JS0ZnuMC008B/esci-product-relevance (1)/experiment_outputs/semantic_bert


In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [
        sys.executable,
        "-u",
        "-m",
        "scripts.run_main_experiment",
        "--config",
        "configs/main_experiment.yaml",
        "--data-dir",
        str(DRIVE_FOLDER),
        "--output-dir",
        str(OUTPUT_DIR),
        "--experiment",
        "semantic_bert",
    ],
    check=True,
    env=env,
)

**6. Validation results**

Gunakan hasil validation untuk membandingkan model saat development.

Metrik utama adalah Macro-F1 dan Weighted-F1. Metrik tambahan adalah Accuracy dan Micro-F1.

Untuk model semantic, checkpoint/best epoch dipilih berdasarkan validation loss, bukan test result.

In [ ]:
ALL_EXPERIMENTS = [
    "lexical_separate",
    "lexical_concatenated",
    "semantic_roberta",
    "semantic_deberta",
]

validation_rows = []

for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"

    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)

    validation_rows.append(
        {
            "experiment": experiment,
            **result["validation"],
        }
    )

validation_results = pd.DataFrame(validation_rows)

display(
    validation_results[
        [
            "experiment",
            "macro_f1",
            "weighted_f1",
            "accuracy",
            "micro_f1",
        ]
    ].sort_values("macro_f1", ascending=False)
)

**7. Final evaluation**

Jalankan bagian ini hanya setelah semua keputusan development dikunci.

Tahap ini adalah satu-satunya tahap yang membaca final representative TEST subset sebanyak 30.551 baris.

Final metric berasal eksklusif dari test tersebut.

In [ ]:
for experiment in ALL_EXPERIMENTS:
    print(f"\n{'=' * 70}")
    print(f"Final evaluation: {experiment}")
    print(f"{'=' * 70}")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "scripts.run_main_experiment",
            "--config",
            "configs/main_experiment.yaml",
            "--data-dir",
            str(DRIVE_FOLDER),
            "--output-dir",
            str(OUTPUT_DIR),
            "--experiment",
            experiment,
            "--final-evaluation",
        ],
        check=True,
    )

**8.Final test metrics**

Metrik berikut berasal hanya dari 30.551 representative official TEST rows:

- Macro-F1;
- Weighted-F1;
- Accuracy;
- Micro-F1;
- F1 setiap kelas E/S/C/I;
- confusion matrix.

In [ ]:
final_rows = []

for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"

    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)

    if not result["final_evaluation_ran"]:
        raise RuntimeError(
            f"Final evaluation belum dijalankan: {experiment}"
        )

    final_rows.append(
        {
            "experiment": experiment,
            **result["final_test"],
        }
    )

final_results = pd.DataFrame(final_rows)

display(
    final_results[
        [
            "experiment",
            "macro_f1",
            "weighted_f1",
            "accuracy",
            "micro_f1",
        ]
    ].sort_values("macro_f1", ascending=False)
)

**9.Per-class F1 dan confusion matrix**

Bagian ini dipakai untuk membandingkan error pattern antar pendekatan lexical dan semantic.

Interpretasi hasil adalah perbandingan performa yang diamati, bukan klaim sebab-akibat.

In [ ]:
for experiment in ALL_EXPERIMENTS:
    result_path = OUTPUT_DIR / experiment / "results.json"

    with result_path.open(encoding="utf-8") as file:
        result = json.load(file)

    final_test = result["final_test"]

    print(f"\n{'=' * 70}")
    print(f"Experiment: {experiment}")
    print(f"{'=' * 70}")

    per_class_f1 = pd.DataFrame(
        [
            {
                "label": label,
                "f1_score": score,
            }
            for label, score in final_test["per_class_f1"].items()
        ]
    )
    display(per_class_f1)

    confusion = pd.DataFrame(
        final_test["confusion_matrix"],
        index=["actual_E", "actual_S", "actual_C", "actual_I"],
        columns=["pred_E", "pred_S", "pred_C", "pred_I"],
    )
    display(confusion)